In [8]:
from sklearn.datasets import fetch_openml
daten = fetch_openml("credit-g", version=1, as_frame=True)
df = daten.frame          # .frame = die ganze Tabelle inkl. Zielspalte "class"
df['ausfall'] = df['class'].map({'good': 0, 'bad': 1}).astype(int)
df.head(5)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class,ausfall
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,67,none,own,2,skilled,1,yes,yes,good,0
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,22,none,own,1,skilled,1,none,yes,bad,1
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,49,none,own,1,unskilled resident,2,none,yes,good,0
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,45,none,for free,1,skilled,2,none,yes,good,0
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,53,none,for free,2,skilled,2,none,yes,bad,1


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

y = df['ausfall']
X = df.drop(columns=['class', 'ausfall'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean(), y_test.sum())

(700, 20) (300, 20)
0.3 0.3 90


Zerlegung:
Die Kategorien in den Category-Spalten erstezen durch 0/1 für jede Ausprägung der jeweiligen Kategorien, z.B. Bei einer Kategorie mit 4 Ausprägungen, entstehen vier neue Spalten anstelle der einen alten mit der Information, welche Kategorie gilt für die jeweilige Zeile. Gleiches Vorgehen für X_Train und X_Test. Danach Prüfung, ob die Anzahl der Spalten und der Namen zwischen den beiden Datensätzen passen.

In [10]:
X_train_k = pd.get_dummies(X_train, dtype=int)
X_test_k = pd.get_dummies(X_test, dtype=int)
print(X_train_k.shape)
print(X_test_k.shape)
print(X_train_k.shape[1] == X_test_k.shape[1])
print(list(X_train_k.columns) == list(X_test_k.columns))
print(X_train_k.drop(columns=X_train.select_dtypes('number').columns).sum(axis=1).unique())

(700, 61)
(300, 61)
True
True
[13]


Zerlegung:
Über fit() die Mittelwerte und skalierung der 7 Zahlenspalten auf Basis der Trainingsdaten ausrechnen
Über transform() die standardisierten werte für die Trainingsdaten und separat für die Testdaten berechnen
Dann das np.array wieder in ein df überführen

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = X_train_k.copy()
X_test_s = X_test_k.copy()

num_cols = X_train.select_dtypes(include='number').columns

X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = scaler.transform(X_test[num_cols])

print(X_train_s.shape, X_test_s.shape)
print(y_train.shape, y_test.shape)
print(X_train_s[num_cols].mean().round(3))
print(X_train_s[num_cols].std().round(2))
print(X_test_s[num_cols].mean().round(3))
print(X_train_k['duration'].head(3))
print(scaler.mean_)
print(X_train_s['housing_own'].head(3))

(700, 61) (300, 61)
(700,) (300,)
duration                  0.0
credit_amount            -0.0
installment_commitment   -0.0
residence_since           0.0
age                      -0.0
existing_credits          0.0
num_dependents           -0.0
dtype: float64
duration                  1.0
credit_amount             1.0
installment_commitment    1.0
residence_since           1.0
age                       1.0
existing_credits          1.0
num_dependents            1.0
dtype: float64
duration                  0.037
credit_amount             0.088
installment_commitment    0.102
residence_since           0.076
age                       0.024
existing_credits          0.025
num_dependents           -0.033
dtype: float64
10     12
82     18
827    18
Name: duration, dtype: int64
[2.07685714e+01 3.20087286e+03 2.93857143e+00 2.82000000e+00
 3.54685714e+01 1.40285714e+00 1.15857143e+00]
10     0
82     0
827    1
Name: housing_own, dtype: int64


## Stand Tag 63 (Mo 19.10.) — Daten modellfertig

**Kette:** df (1000, 22) → Split → One-Hot-Kodierung → Standardisierung.
Reihenfolge zwingend: erst teilen, dann alles Weitere nur aus dem Trainingsteil lernen — sonst sieht das Modell Testdaten (Datenleck).

**Tabellen:**
- y = Spalte `ausfall` (1000,), 0/1. `class` und `ausfall` aus X entfernt, damit keine Ergebniswerte in den Input-Daten stehen.
- X_train (700, 20), X_test (300, 20), y_train, y_test — aus `train_test_split`, 70/30, `stratify=y` (homogene Datengrundlage: Ausfallquote 0,30 in beiden Teilen, 90 Test-Ausfälle), `random_state=42`.
- X_train_k (700, 61), X_test_k (300, 61) — aus `get_dummies`: 13 Kategorie-Spalten → 54 Null-Eins-Spalten, 7 Zahlenspalten unverändert.
- X_train_s (700, 61), X_test_s (300, 61) — aus `StandardScaler`: 7 Zahlenspalten mit Mittelwert/Streuung aus **Train** umgerechnet (fit nur auf Train, transform auf beide), 54 Null-Eins-Spalten unverändert.

**Modellfertig für morgen:** X_train_s, X_test_s, y_train, y_test.

**Der eine Satz:** `fit` lernt Zahlen und speichert sie mit Unterstrich (`mean_`, `coef_`); `transform` und `predict` benutzen diese Zahlen und geben etwas Neues zurück, ohne etwas zu speichern.

**Gestolpert heute:** (1) geglaubt, `fit` verändere die Daten bzw. speichere Vorhersagen — tut es nicht; (2) Standardisierung auf X_train statt X_train_k aufgesetzt, Kontrollwert 20 statt 61 nicht gegen den Sollwert geprüft.

## Zerlegung Teil 2c — eigene LogistischeRegression auf den Kreditdaten

Ziel: Meine Klasse vom 09.10. lernt aus 700 Krediten mit 61 Merkmalen,
ob ein Kredit ausfällt. Am Ende: Gewichte, Kostenverlauf, Accuracy auf dem Testteil.

1. Tabellen in Zahlen-Arrays verwandeln — meine Klasse rechnet mit Matrizen,
   nicht mit DataFrames. Ergebnis: Xtr (700, 61), ytr (700,), Xte (300, 61), yte (300,).
2. Klasse unverändert übernehmen — sie liest die Spaltenzahl selbst aus X.shape,
   deshalb gilt sie für 2 Merkmale wie für 61.
3. Objekt anlegen mit Lernrate 0,1 und 1000 Schritten, dann fit auf Xtr/ytr.
   Ergebnis: w (61,) — ein Gewicht je Spalte; b eine Zahl; J_verlauf Liste mit 1000 Kosten.
4. Prüfen, ob gelernt wurde: Kosten am Anfang (0,693 = analog zu MÜnzwurf, da auch p = 0.5) gegen Kosten am Ende,
   und ob kein Schritt die Kosten erhöht hat (größte Differenz ≤ 0).
5. Vorhersage auf Xte, Vergleich mit yte, Anteil der Treffer = Accuracy.
   Maßstab: 0,70 wäre „immer good" — alles darüber hat das Modell gelernt.

In [12]:
import numpy as np
Xtr = X_train_s.values
ytr = y_train.values
Xte = X_test_s.values
yte = y_test.values

class LogistischeRegression:
    def __init__(self, alpha=0.1, n_iter=1000):
        # Hyperparameter speichern; self.w = None, self.b = None, self.J_verlauf = []
        self.alpha = alpha
        self.n_iter = n_iter
        self.w = None
        self.b = None
        self.J_verlauf = []

    def sigmoid(self, z):
        # 1 / (1 + np.exp(-z))
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        m, n = X.shape
        w = np.zeros(n) 
        b = 0.0
        for i in range(self.n_iter):
            p = self.sigmoid(X @ w + b)
            err = p - y
            w -= self.alpha * (X.T @ err) / m
            b -= self.alpha * err.mean()
            J = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
            self.J_verlauf.append(J)
        self.w = w
        self.b = b

        return self

    def predict_proba(self, X):
        return self.sigmoid(X @ self.w + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


eigen = LogistischeRegression(alpha=0.1, n_iter=1000).fit(Xtr, ytr)
eigen2 = LogistischeRegression(alpha=0.5, n_iter=1000).fit(Xtr, ytr)
d = np.diff(eigen.J_verlauf)
#print(eigen.w.shape, eigen.w, round(eigen.b, 3), round(eigen.J_verlauf[0], 3), round(eigen.J_verlauf[-1], 3))
#print(round(d.max(), 6), np.all(d <= 0))

print(round(eigen2.J_verlauf[9], 3), round(eigen2.J_verlauf[99], 3),round(eigen2.J_verlauf[-1], 3))
print(round(eigen.J_verlauf[9], 3), round(eigen.J_verlauf[99], 3),round(eigen.J_verlauf[-1], 3))
print(eigen2.w.shape, round(eigen2.b, 3))
print((eigen2.predict(Xte) == yte).mean())
print((eigen2.predict(Xtr) == ytr).mean())
print((eigen.predict(Xte) == yte).mean())
print((eigen.predict(Xtr) == ytr).mean())


0.531 0.461 0.447
0.586 0.5 0.454
(61,) -0.357
0.77
0.7914285714285715
0.7766666666666666
0.7757142857142857


## Zerlegung Teil 2d — sklearn LogisticRegression auf denselben Daten

Ziel: Dasselbe Modell aus der Bibliothek, dann Vergleich mit meiner Klasse —
gleiche Vorzeichen? gleiche Accuracy?

1. Klasse LogisticRegression aus sklearn.linear_model laden (import) und ein Objekt sk
   anlegen mit Vorgabe „höchstens 1000 Schritte". Ergebnis: leeres Modell, noch ohne coef_.
2. fit auf X_train_s (700, 61) und y_train (700,) — DataFrames gehen direkt, kein .values.
   fit speichert im Objekt: coef_ (1, 61), intercept_ eine Zahl, n_iter_ die gebrauchten Schritte.
   Rückgabe ist das Objekt selbst.
3. score auf X_test_s (300, 61) und y_test (300,) → eine Zahl, die Accuracy.
   Daneben stellen: eigen 0,777 und „immer good" 0,70.
4. Vergleichstabelle: Zeilen = 61 Spaltennamen aus X_train_s.columns,
   Spalte eigen = eigen.w (61,), Spalte sklearn = coef_[0] (61,) — die [0] holt die Zeile
   aus der (1, 61)-Form. Ergebnis: DataFrame (61, 2).
5. Vorzeichen beider Spalten vergleichen, Treffer zählen → eine Zahl von 61.
   Erwartung: fast alle gleich, Beträge verschieden (sklearn regularisiert, C=1.0).

In [13]:
import numpy as np
from sklearn.linear_model import LogisticRegression

sk = LogisticRegression(max_iter=1000)
sk.fit(X_train_s, y_train)

print("Koeffizienten_Form:", sk.coef_.shape)
print("Intercept:", sk.intercept_)
print("Klassen:", sk.classes_)
print("Iterationen:", sk.n_iter_)
print("Accuracy Train:", sk.score(X_train_s, y_train))
print("Accuracy Test:", sk.score(X_test_s, y_test))

vergleich = pd.DataFrame({'eigen': eigen.w, 'sklearn': sk.coef_[0]}, index=X_train_s.columns)
same_sign = (np.sign(vergleich['eigen']) == np.sign(vergleich['sklearn'])).sum()
rel_sign = (np.sign(vergleich['eigen']) == np.sign(vergleich['sklearn'])).mean()
print(vergleich)
print(same_sign, rel_sign)

abweichler = vergleich[np.sign(vergleich['eigen']) != np.sign(vergleich['sklearn'])]
print(abweichler.round(3))

Koeffizienten_Form: (1, 61)
Intercept: [-1.55786068]
Klassen: [0 1]
Iterationen: [53]
Accuracy Train: 0.79
Accuracy Test: 0.7833333333333333
                           eigen   sklearn
duration                0.353234  0.358200
credit_amount           0.185135  0.202612
installment_commitment  0.342222  0.351279
residence_since        -0.020438 -0.017388
age                    -0.094100 -0.089644
...                          ...       ...
job_skilled            -0.063552  0.071423
own_telephone_none      0.018506  0.128968
own_telephone_yes      -0.228837 -0.130939
foreign_worker_no      -0.454997 -0.650275
foreign_worker_yes      0.244665  0.648304

[61 rows x 2 columns]
55 0.9016393442622951
                              eigen  sklearn
purpose_furniture/equipment  -0.090    0.012
savings_status_500<=X<1000   -0.064    0.085
personal_status_male mar/wid -0.035    0.029
job_unemp/unskilled non res   0.020   -0.092
job_unskilled resident       -0.043    0.091
job_skilled                 

Zerlegung zu Confusion Matrix:
1. Vier Zahlen der Vierfeldertafel:  
   Feld1: Tatsächlich gut & Modell gut
   => y_test == 0 & predict == 0 mit ((y == 0) & (pred == 0)).sum()
   Feld2: Tatsächlich Ausfall & Modell Ausfall
   => y_test == 1 & predict == 1 mit ((y == 1) & (pred == 1)).sum()
   Feld3: Modell gut & Tatsächlich Ausfall
   => predict == 0 & y_test == 1 mit ((y == 1) & (pred == 0)).sum()
   Feld4: Modell Ausfall & Tatsächlich gut
   => predict == 1 und y_test == 0 mit ((y == 0) & (pred == 1)).sum()
2. Anzahl richtig vorhergesagter Ausfälle / Anzahl gesamte tatsächliche. 
   Ausfälle gesamt (Recall); Anzahl richtig vorhergesagter Ausfällt / Anzahl vorhergesagter Ausfälle gesamt (Precision)
3. Genauigkeit/Accuracy über beide Wege

In [15]:
# basierend auf eigen:
pred_eigen = eigen.predict(Xte)
field_a_eig = ((yte == 0) & (pred_eigen == 0)).sum()
field_b_eig = ((yte == 1) & (pred_eigen == 1)).sum()
field_c_eig = ((yte == 1) & (pred_eigen == 0)).sum()
field_d_eig = ((yte == 0) & (pred_eigen == 1)).sum()
print(f'eig-model: Real-good Pred-good: {field_a_eig}, Real-bad Pred-bad: {field_b_eig}, Real-bad Pred-good: {field_c_eig}, Real-good Pred-bad : {field_d_eig}')

# basierend auf sklearn:
pred_sk = sk.predict(X_test_s)
p_ausfall = sk.predict_proba(X_test_s)[:, 1]
pred_03 = (p_ausfall >= 0.3).astype(int)
field_a_sk = ((y_test == 0) & (pred_03 == 0)).sum()
field_b_sk = ((y_test == 1) & (pred_03 == 1)).sum()
field_c_sk = ((y_test == 1) & (pred_03 == 0)).sum()
field_d_sk = ((y_test == 0) & (pred_03 == 1)).sum()
print(f'sk-model: Real-good Pred-good: {field_a_sk}, Real-bad Pred-bad: {field_b_sk}, Real-bad Pred-good: {field_c_sk}, Real-good Pred-bad : {field_d_sk}')


recall_eigen = ((pred_eigen == 1) & (yte == 1)).sum() / (yte == 1).sum()
recall_sk = ((pred_03 == 1) & (y_test == 1)).sum() / (y_test == 1).sum()
precision_eigen = ((pred_eigen == 1) & (yte == 1)).sum() / (pred_eigen == 1).sum()
precision_sk = ((pred_03 == 1) & (y_test == 1)).sum() / (pred_03 == 1).sum()

print(f'Recall-eigen: {recall_eigen:.1%}, Recall_sk: {recall_sk:.1%}')
print(f'Precision-eigen: {precision_eigen:.1%}, Precision_sk: {precision_sk:.1%}')

from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score, precision_score

print(f'Confusion Matrix: {confusion_matrix(y_test, pred_03)}')
print(f'Recall-Score: {recall_score(yte, pred_eigen):.1%}, Precision-Score: {precision_score(yte, pred_eigen):.1%}')

maske = (pred_03 == 0) & (y_test == 1)
ne_ausfall = X_test.loc[maske, 'credit_amount'].sum() * 0.85
print(f"Nicht erkannte Ausfälle des Modells: {ne_ausfall:,.0f}")
print(f"Gesamt-Kreditsumme: {X_test['credit_amount'].sum():,.0f}")

maske2 = (y_test == 1)
gesamt_ausfall = X_test.loc[maske2, 'credit_amount'].sum() * 0.85
print(f"Gesamtausfall: {gesamt_ausfall:,.0f}")


model_adv_abs = gesamt_ausfall - ne_ausfall
print(f'Der Vorteil mit Modell vs. ohne Modell (abs): {model_adv_abs:,.0f}')
model_adv_rel = model_adv_abs / gesamt_ausfall
print(f'Der Vorteil mit Modell vs. ohne Modell (rel): {model_adv_rel:.1%}')



eig-model: Real-good Pred-good: 185, Real-bad Pred-bad: 48, Real-bad Pred-good: 42, Real-good Pred-bad : 25
sk-model: Real-good Pred-good: 151, Real-bad Pred-bad: 66, Real-bad Pred-good: 24, Real-good Pred-bad : 59
Recall-eigen: 53.3%, Recall_sk: 73.3%
Precision-eigen: 65.8%, Precision_sk: 52.8%
Confusion Matrix: [[151  59]
 [ 24  66]]
Recall-Score: 53.3%, Precision-Score: 65.8%
Nicht erkannte Ausfälle des Modells: 95,402
Gesamt-Kreditsumme: 1,030,647
Gesamtausfall: 342,124
Der Vorteil mit Modell vs. ohne Modell (abs): 246,722
Der Vorteil mit Modell vs. ohne Modell (rel): 72.1%


## Stand Tag 64 (Di 20.10.) — zwei Modelle trainiert, Accuracy gegen Nichtstun

**Was steht:**
- `eigen` — meine Klasse `LogistischeRegression` (alpha 0,1, 1000 Schritte), trainiert auf
  `Xtr` (700, 61) / `ytr` (700,) — Arrays aus `X_train_s.values`, `y_train.values`.
  Gelernt: `eigen.w` (61,), `eigen.b` = −0,21 (eine Zahl), `eigen.J_verlauf` (Liste, 1000 Kosten).
- `sk` — sklearn `LogisticRegression(max_iter=1000)`, trainiert direkt auf den DataFrames
  `X_train_s` / `y_train`. Gelernt (Unterstrich = von fit gespeichert): `coef_` (1, 61),
  `intercept_` = −1,56, `n_iter_` = 53 Schritte bis Konvergenz.
- `vergleich` (61, 2) — beide Gewichtsvektoren nebeneinander, Zeilen = Spaltennamen.

**Zahlen, die man sich merkt:**
- Kosten Start 0,693 = ln 2 = Münzwurf (w = 0 → jede Wahrscheinlichkeit 0,5); Ende 0,454;
  kein Schritt bergauf (größte Differenz −8e-06).
- Accuracy Test: „immer good" 0,70 (210/300) · eigen 0,777 (233) · sklearn 0,783 (235).
  Modelle untereinander: kein Unterschied (2 Kredite). Gegen Nichtstun: ~23 Kredite mehr richtig.
  Aber: „immer good" erkennt 0 der 90 Ausfälle — das misst Accuracy nicht → Do Confusion Matrix.
- Vorzeichen: 55 von 61 gleich (0,90); die 6 Abweichler haben Beträge ≤ 0,09 — keine inhaltliche
  Meinungsverschiedenheit. Beträge verschieden, weil sklearn regularisiert (C = 1,0) und
  mit L-BFGS bis zur Konvergenz rechnet statt 1000 Gradientenschritte.
- Erste sieben Gewichte = die sieben Zahlenspalten: Laufzeit +0,35, Rate +0,34, bestehende
  Kredite +0,21, Betrag +0,19 → riskanter; Alter −0,09 → sicherer. Deutung am Do.

**Was gewackelt hat (Denkmodell-Fehler heute: 1):**
- Behälter vs. Inhalt: `np.sign('eigen')` → TypeError; `'eigen'` ist der Name, `vergleich['eigen']`
  die Spalte. Gleiches Muster wie `b.shape` (b ist eine Zahl, keine Form) — Tastversuch, kein Fehler.
- `coef_` ist (1, 61), zweidimensional mit einer Zeile; `pd.DataFrame` will (61,) → `coef_[0]` oder `.ravel()`.
  Grund: sklearn hält eine Zeile je Klasse bereit, bei zwei Klassen bleibt es eine.
- Name `df` war schon die Kredittabelle → Vergleichstabelle heißt `vergleich`. Ein Name pro Behälter.
- Zerlegung zweimal zu dünn (übersprungen / ohne Formen) — Benchmark: Eingabe → was passiert → Ergebnis mit Form.

**Offen bis Do 22.10.:** Confusion Matrix, Precision/Recall für die 90 Ausfälle · `predict_proba`,
Schwelle 0,5 vs. 0,3 · Deutung der drei größten Gewichte · Feynman-Test · Abschluss-Stand vor dem Urlaub.


Zerlegung zu Confusion Matrix:
1. Vier Zahlen der Vierfeldertafel:  
   Feld1: Tatsächlich gut & Modell gut
   => y_test == 0 & predict == 0 mit ((y == 0) & (pred == 0)).sum()
   Feld2: Tatsächlich Ausfall & Modell Ausfall
   => y_test == 1 & predict == 1 mit ((y == 1) & (pred == 1)).sum()
   Feld3: Modell gut & Tatsächlich Ausfall
   => predict == 0 & y_test == 1 mit ((y == 1) & (pred == 0)).sum()
   Feld4: Modell Ausfall & Tatsächlich gut
   => predict == 1 und y_test == 0 mit ((y == 0) & (pred == 1)).sum()
2. Anzahl richtig vorhergesagter Ausfälle / Anzahl gesamte tatsächliche. 
   Ausfälle gesamt (Recall); Anzahl richtig vorhergesagter Ausfällt / Anzahl vorhergesagter Ausfälle gesamt (Precision)
3. Genauigkeit/Accuracy über beide Wege